# Coop Case — Q4.3: Do Customers Repeatedly Buy the Same Type of Item — and Can We Target Discounts to Push Higher-Margin Products?

**Question:** Does the same customer recurringly buy the same type of item? If so, does it make
sense to target different customers with different discounts to shift them toward more profitable
products?

Sub-questions:
- How concentrated is each household's spending across categories -- repeat/loyal buyers vs. diverse shoppers?
- Which categories are actually the most/least profitable (margin %, not just revenue)?
- Are "loyal" customers disproportionately anchored in low-margin categories -- i.e. is there a real
  targeting opportunity?


## 1. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)


In [ ]:
DATA_PATH = "2months_v2/rl_2months.csv"

dtypes = {
    "receiptKey": "int64",
    "hourOfDay": "int8",
    "minuteOfHour": "int8",
    "quantity": "float32",
    "lineItemAmount": "float32",
    "lineItemAmountExclVat": "float32",
    "discountAmountExclVat": "float32",
    "lineItemCostExclVat": "float32",
    "CoopOnlineYN": "category",
    "store": "category",
    "customerId": "Int64",
    "householdId": "Int64",
    "MOSAICGroup": "category",
    "MOSAICGroupDescription": "category",
    "MOSAICType": "category",
    "MOSAICTypeDescription": "category",
    "DominantBuyingPowerClass": "category",
    "ItemID": "int64",
    "ItemSubSegmentName": "category",
    "ItemSubSegmentID": "Int64",
    "ItemSegmentName": "category",
    "ItemSegmentID": "Int64",
    "ItemSubCategoryName": "category",
    "ItemSubCategoryID": "Int64",
    "ItemCategoryName": "category",
    "ItemCategoryID": "Int64",
    "ItemCategoryTeamName": "category",
    "ItemCategoryTeamID": "Int64",
    "ItemCategoryGroupName": "category",
    "ItemCategoryGroupID": "Int64",
    "ItemCategoryAreaName": "category",
    "ItemCategoryAreaID": "Int64",
    "Brand": "category",
    # Stored as floats in the source file (0.0 / 1.0), not clean ints -- pandas
    # won't safely downcast float64 -> int8 during read_csv, so keep as float32.
    "eko": "float32",
    "organic": "float32",
    "krav": "float32",
    "fair_trade": "float32",
    "msc": "float32",
    "no_lactose": "float32",
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=["DayDate"],
)
df["profit"] = df["lineItemAmountExclVat"] - df["lineItemCostExclVat"]

print(df.shape)
df.head()

## 2. Household x category spend, and a concentration index (HHI)

Uses a Herfindahl-Hirschman-style index per household: sum of each category's squared share of
that household's total spend. HHI = 1.0 would mean 100% of spend in one category; low HHI means a
diverse basket across many categories.

**Data-quality note:** a handful of households have return/correction lines that make a category's
revenue negative, which can push a category's *share* above 1 or below 0 for that household (and
HHI above the usual 0-1 range) -- a real feature of the data (returns), not a bug. We don't hide it:
the summary below shows the raw distribution including this effect, and we flag it in the takeaways.


In [ ]:
cat_df = df.dropna(subset=["ItemCategoryTeamName"])

hh_cat = cat_df.groupby(["householdId", "ItemCategoryTeamName"], observed=True)["lineItemAmountExclVat"].sum().reset_index()
hh_total = hh_cat.groupby("householdId")["lineItemAmountExclVat"].sum().rename("total")
hh_cat = hh_cat.merge(hh_total, on="householdId")
hh_cat = hh_cat[hh_cat["total"] > 0]
hh_cat["share"] = hh_cat["lineItemAmountExclVat"] / hh_cat["total"]

hhi = hh_cat.groupby("householdId")["share"].apply(lambda s: (s ** 2).sum()).rename("hhi")

print(hhi.describe().round(3))
print()
print(f"Share of households with HHI > 0.5 (highly concentrated in ~1-2 categories): {(hhi > 0.5).mean():.1%}")
print(f"Share of households with HHI < 0.15 (very diverse basket): {(hhi < 0.15).mean():.1%}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
hhi.clip(upper=1.0).plot(kind="hist", bins=40, ax=ax, color="#22B573")
ax.set_title("Distribution of household category-concentration (HHI, clipped at 1.0)")
ax.set_xlabel("HHI (higher = more concentrated in fewer categories)")
ax.set_ylabel("Number of households")
plt.tight_layout()
plt.show()


## 3. Category profitability (margin %, not just revenue)

In [ ]:
cat_margin = cat_df.groupby("ItemCategoryTeamName", observed=True).agg(
    revenue=("lineItemAmountExclVat", "sum"),
    profit=("profit", "sum"),
)
cat_margin = cat_margin[cat_margin["revenue"] > 10_000]  # drop noisy tiny categories
cat_margin["margin_pct"] = cat_margin["profit"] / cat_margin["revenue"] * 100
cat_margin = cat_margin.sort_values("margin_pct", ascending=False)

print("Top 8 highest-margin categories:")
print(cat_margin.head(8).round(2))
print()
print("Bottom 8 lowest-margin categories:")
print(cat_margin.tail(8).round(2))


In [ ]:
top_bottom = pd.concat([cat_margin.head(8), cat_margin.tail(8)])
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#22B573"] * 8 + ["#E8734A"] * 8
ax.barh(top_bottom.index, top_bottom["margin_pct"], color=colors)
ax.set_title("Category margin %: top 8 (green) vs. bottom 8 (orange)")
ax.set_xlabel("Margin %")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 4. Are "loyal" (concentrated) households anchored in low-margin categories?

For each household with HHI > 0.5 (genuinely repeat-buying one category), find their single
top category and its margin -- then check how many sit below the median category margin.


In [ ]:
idx = hh_cat.groupby("householdId")["share"].idxmax()
top_cat = hh_cat.loc[idx][["householdId", "ItemCategoryTeamName", "share"]].rename(
    columns={"ItemCategoryTeamName": "top_category", "share": "top_category_share"}
)
top_cat = top_cat.merge(cat_margin[["margin_pct"]], left_on="top_category", right_index=True, how="left")
top_cat["hhi"] = top_cat["householdId"].map(hhi)

concentrated = top_cat[top_cat["hhi"] > 0.5].dropna(subset=["margin_pct"])
median_margin = cat_margin["margin_pct"].median()
below_median = concentrated[concentrated["margin_pct"] < median_margin]

print(f"Households highly concentrated in one category (HHI > 0.5): {concentrated.shape[0]:,}")
print(f"Median category margin (all categories): {median_margin:.1f}%")
print(f"Median margin of concentrated households' own top category: {concentrated['margin_pct'].median():.1f}%")
print(f"Concentrated households whose top category is BELOW median margin: "
      f"{below_median.shape[0]:,} ({below_median.shape[0] / concentrated.shape[0]:.1%})")


In [ ]:
below_median["top_category"].value_counts().head(10).plot(
    kind="barh", figsize=(8, 5), color="#E8734A", title="Most common 'anchor' category among\nconcentrated households sitting below median margin"
)
plt.xlabel("Number of households")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 5. Takeaways

- **Most customers are NOT single-category loyalists.** 46.4% of households have HHI < 0.15 (very
  diverse category mix); only 13.1% (~2,667 households) are highly concentrated (HHI > 0.5, roughly
  1-2 categories dominating their spend). So "recurring same-item buying" is real, but it's a
  minority behavior, not the norm -- targeted category-based discounting has a real but bounded
  audience.

- **Category margins vary enormously** -- from slightly negative (non-sale/service categories) up
  to ~52% (Hemmet/home goods). Notably, two high-volume "everyday staple" categories are among the
  **lowest-margin** in the whole assortment: **KÖTT (meat) at 9.76%** and **FRUKT, BÄR
  (fruit & berries) at 4.79%** -- both far below categories like FÄRDIGMAT (ready meals, 41.50%) or
  SKÖNHET, HÄLSA (beauty & health, 39.28%). This matters directly for targeting strategy.

- **The targeting opportunity is real and sizeable**: of the 2,667 highly-concentrated ("loyal")
  households, **41.9%** have their anchor category sitting *below* the median category margin.
  Nearly half of Coop's most predictable, repeat-pattern customers are anchored in relatively
  low-margin categories -- exactly the group where a well-designed nudge (cross-sell discount toward
  a higher-margin adjacent category, e.g. from raw meat toward a higher-margin ready-meal version, or
  bundling fruit purchases with a beauty/health promo) has the clearest hypothesis behind it.

- **Important limitation, stated plainly**: this is a *targeting feasibility* finding, not proof
  that targeted discounts will work. We have no experimental variation in discount exposure in this
  dataset (no natural A/B test), so we can't estimate how much a nudge would actually shift behavior
  -- only that the population who'd be worth testing it on is identifiable and sizeable. The
  recommended next step is a controlled pilot: offer a matched sample of low-margin-anchored,
  high-concentration households a targeted higher-margin substitute discount, and measure category-mix
  shift against a held-out control group.

- **A data-quality note surfaced while building this**: households with return/correction lines can
  push their category HHI above the normal 0-1 range (a category's revenue can go negative,
  making its "share" negative or another category's share exceed 1). This is a real feature of
  returns in the data, not a computation bug -- worth being aware of if this HHI metric is reused
  elsewhere, since it isn't a strictly bounded 0-1 index for every household.
